In [ ]:
#CNN(original version)
import os
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import random

ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, "data")
MODEL_DIR = os.path.join(ROOT, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'[^A-Za-z0-9\u4e00-\u9fff ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def build_text_column(df: pd.DataFrame, duplicate_title=True, max_len=3000):
    title = df["title"].fillna("").map(normalize_text)
    content = df["content"].fillna("").map(normalize_text)
    if duplicate_title:
        text = title + " [SEP] " + title + " " + content
    else:
        text = title + " [SEP] " + content
    return text.str.slice(0, max_len)

def main():
    ds = load_dataset("sogou_news")
    train_df = ds["train"].to_pandas()
    test_df = ds["test"].to_pandas()

    train_df.to_csv(os.path.join(DATA_DIR, "sogou_train.csv"), index=False)
    test_df.to_csv(os.path.join(DATA_DIR, "sogou_test.csv"), index=False)

    train_texts = build_text_column(train_df)
    test_texts = build_text_column(test_df)

    y_train_full = train_df["label"].astype(int).values
    y_test = test_df["label"].astype(int).values

    X_train, X_val, y_train, y_val = train_test_split(
        train_texts, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42
    )

    tokenizer = Tokenizer(num_words=50000, oov_token="[UNK]")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)
    X_test_seq = tokenizer.texts_to_sequences(test_texts)

    max_len = 1000
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

    vocab_size = min(50000, len(tokenizer.word_index) + 1)

    model = Sequential([
        Embedding(vocab_size, 256, input_length=max_len),
        Conv1D(128, 5, activation="relu"),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(len(np.unique(y_train_full)), activation="softmax")
    ])

    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    model.summary()

    early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

    model.fit(
        X_train_pad, y_train,
        epochs=10,
        batch_size=64,
        validation_data=(X_val_pad, y_val),
        callbacks=[early_stop],
        verbose=1
    )

    y_val_pred = np.argmax(model.predict(X_val_pad), axis=1)
    print("\nValidation Report")
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    y_test_pred = np.argmax(model.predict(X_test_pad), axis=1)
    print("\nTest Report")
    print(classification_report(y_test, y_test_pred))

    model_path = os.path.join(MODEL_DIR, "cnn_text_classifier.h5")
    model.save(model_path)
    print("\nModel saved to:", model_path)

    return test_df, y_test, y_test_pred

if __name__ == "__main__":
    test_df, y_test, y_test_pred = main()

label_map = {
    0: "Sports",
    1: "Finance",
    2: "Entertainment",
    3: "Automobile",
    4: "Technology"
}

samples = random.sample(range(len(test_df)), 5)
for i in samples:
    print("\n")
    print("Title:", test_df.iloc[i]['title'][:50])
    print("Content:", test_df.iloc[i]['content'][:120])
    print("True Label:", label_map.get(int(y_test[i]), str(y_test[i])))
    print("Predicted Label:", label_map.get(int(y_test_pred[i]), str(y_test_pred[i])))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

...

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 234s 35ms/step - accuracy: 0.8775 - loss: 0.3746 - val_accuracy: 0.9516 - val_loss: 0.1520
Epoch 2/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 221s 35ms/step - accuracy: 0.9454 - loss: 0.1837 - val_accuracy: 0.9576 - val_loss: 0.1394
Epoch 3/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 221s 35ms/step - accuracy: 0.9518 - loss: 0.1602 - val_accuracy: 0.9581 - val_loss: 0.1368
Epoch 4/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 221s 35ms/step - accuracy: 0.9552 - loss: 0.1480 - val_accuracy: 0.9590 - val_loss: 0.1323
Epoch 5/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 221s 35ms/step - accuracy: 0.9574 - loss: 0.1371 - val_accuracy: 0.9589 - val_loss: 0.1357
Epoch 6/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 221s 35ms/step - accuracy: 0.9589 - loss: 0.1321 - val_accuracy: 0.9605 - val_loss: 0.1348
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step

Validation Report
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      9000
           1       0.93      0.


Test Report
              precision    recall  f1-score   support

           0       0.97      0.96      0.97     12000
           1       0.93      0.95      0.94     12000
           2       0.97      0.97      0.97     12000
           3       0.95      0.98      0.96     12000
           4       0.97      0.92      0.95     12000

    accuracy                           0.96     60000
   macro avg       0.96      0.96      0.96     60000
weighted avg       0.96      0.96      0.96     60000


Model saved to: /content/models/cnn_text_classifier.h5


Title:  a4o di2 na4 sa4i xia3o jia1 he2ng sa3o ju4 re2n s
Content: te2ng xu4n ti3 yu4 xu4n   be3i ji1ng shi2 jia1n 6 yue4 16 ri4 xia1o xi1 , zo3ng jia3ng ji1n e2 we2i 34.9 wa4n o1u yua2n 
True Label: Sports
Predicted Label: Sports


Title:  gua1n xi1n ya2n xi3 hua1n na2n yo3u yo3u dia3n pa
Content: ga1ng fa1 xi2ng xi1n zhua1n ji2 < zui4 a4i  Jade I> di2 gua1n xi1n ya2n ji4n ri4 da4o gua3ng zho1u xua1n chua2n , ta3n y
True Label: Enterta

In [ ]:
#TextCNN + BatchNormalization + update parameters
import os
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,Conv1D,GlobalMaxPooling1D,Dense,Dropout,Concatenate,BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
import random

ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT,"data")
MODEL_DIR = os.path.join(ROOT,"models")
os.makedirs(DATA_DIR,exist_ok=True)
os.makedirs(MODEL_DIR,exist_ok=True)

def normalize_text(s:str) ->str:
    if not isinstance(s,str):
        return ""
    s = re.sub(r'https?://\S+|www\.\S+',' ',s)
    s = re.sub(r'[^A-Za-z0-9\u4e00-\u9fff ]+',' ',s)
    s = re.sub(r'\s+',' ',s).strip()
    return s

def build_text_column(df: pd.DataFrame,duplicate_title=True,max_len=3000):
    title = df["title"].fillna("").map(normalize_text)
    content = df["content"].fillna("").map(normalize_text)
    if duplicate_title:
        text = title + " [SEP] " + title + " " + content
    else:
        text = title + " [SEP] " + content
    return text.str.slice(0,max_len)

def main():
    ds = load_dataset("sogou_news")
    train_df = ds["train"].to_pandas()
    test_df = ds["test"].to_pandas()

    train_df.to_csv(os.path.join(DATA_DIR, "sogou_train.csv"), index=False)
    test_df.to_csv(os.path.join(DATA_DIR, "sogou_test.csv"), index=False)

    train_texts = build_text_column(train_df)
    test_texts = build_text_column(test_df)

    y_train_full = train_df["label"].astype(int).values
    y_test = test_df["label"].astype(int).values

    X_train, X_val, y_train, y_val = train_test_split(
        train_texts, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42
    )

    tokenizer = Tokenizer(num_words=50000, oov_token="[UNK]")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)
    X_test_seq = tokenizer.texts_to_sequences(test_texts)

    max_len = 1000
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

    vocab_size = min(50000, len(tokenizer.word_index) + 1)
    embedding_dim = 128
    num_classes = len(np.unique(y_train_full))
    filter_sizes = [3, 4, 5]
    num_filters = 256

    inputs = Input(shape=(max_len,))
    embedding = Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)

    conv_outputs = []
    for fs in filter_sizes:
        conv = Conv1D(filters=num_filters, kernel_size=fs, activation='relu')(embedding)
        conv = BatchNormalization()(conv)
        pool = GlobalMaxPooling1D()(conv)
        conv_outputs.append(pool)

    merged = Concatenate()(conv_outputs)
    drop = Dropout(0.3)(merged)
    dense = Dense(128, activation="relu")(drop)
    drop2 = Dropout(0.3)(dense)

    outputs = Dense(num_classes, activation="softmax")(drop2)

    model = Model(inputs, outputs)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    model.summary()

    early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

    model.fit(
        X_train_pad, y_train,
        epochs=10,
        batch_size=64,
        validation_data=(X_val_pad, y_val),
        callbacks=[early_stop],
        verbose=1
    )

    y_val_pred = np.argmax(model.predict(X_val_pad), axis=1)
    print("\nValidation Report")
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    y_test_pred = np.argmax(model.predict(X_test_pad), axis=1)
    print("\nTest Report")
    print(classification_report(y_test, y_test_pred))

    model_path = os.path.join(MODEL_DIR, "textcnn_optimized.h5")
    model.save(model_path)
    print("\nModel saved to:", model_path)

    return test_df, y_test, y_test_pred

if __name__ == "__main__":
    test_df, y_test, y_test_pred = main()

label_map = {
    0: "Sports",
    1: "Finance",
    2: "Entertainment",
    3: "Automobile",
    4: "Technology"
}

samples = random.sample(range(len(test_df)), 5)
for i in samples:
    print("\n")
    print("Title:", test_df.iloc[i]['title'][:50])
    print("Content:", test_df.iloc[i]['content'][:120])
    print("True Label:", label_map.get(int(y_test[i]), str(y_test[i])))
    print("Predicted Label:", label_map.get(int(y_test_pred[i]), str(y_test_pred[i])))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1000, 128) │  6,400,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 998, 256)  │     98,560 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 997, 256)  │    131,328 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 996, 256)  │    164,096 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 998, 256)  │      1,024 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 997, 256)  │      1,024 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 996, 256)  │      1,024 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 768)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 768)       │          0 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     98,432 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 5)         │        645 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,896,133 (26.31 MB)

 Trainable params: 6,894,597 (26.30 MB)

 Non-trainable params: 1,536 (6.00 KB)

Epoch 1/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 349s 53ms/step - accuracy: 0.8929 - loss: 0.3314 - val_accuracy: 0.9592 - val_loss: 0.1376
Epoch 2/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 329s 52ms/step - accuracy: 0.9535 - loss: 0.1523 - val_accuracy: 0.9591 - val_loss: 0.1334
Epoch 3/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 380s 52ms/step - accuracy: 0.9582 - loss: 0.1357 - val_accuracy: 0.9610 - val_loss: 0.1350
Epoch 4/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 329s 52ms/step - accuracy: 0.9619 - loss: 0.1221 - val_accuracy: 0.9622 - val_loss: 0.1278
Epoch 5/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 329s 52ms/step - accuracy: 0.9662 - loss: 0.1071 - val_accuracy: 0.9646 - val_loss: 0.1223
Epoch 6/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 330s 52ms/step - accuracy: 0.9695 - loss: 0.0971 - val_accuracy: 0.9650 - val_loss: 0.1286
Epoch 7/10
6329/6329 ━━━━━━━━━━━━━━━━━━━━ 329s 52ms/step - accuracy: 0.9715 - loss: 0.0882 - val_accuracy: 0.9654 - val_loss: 0.1331
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step

Validation Report
      


Test Report
              precision    recall  f1-score   support

           0       0.98      0.96      0.97     12000
           1       0.94      0.95      0.95     12000
           2       0.97      0.98      0.98     12000
           3       0.97      0.98      0.97     12000
           4       0.96      0.95      0.95     12000

    accuracy                           0.96     60000
   macro avg       0.96      0.96      0.96     60000
weighted avg       0.96      0.96      0.96     60000


Model saved to: /content/models/textcnn_optimized.h5


Title:  fe4n me4n zhu3 shua4i zhe3ng su4 zuo4 fe1ng   ta2
Content: te2ng xu4n ti3 yu4 xu4n   co2ng za4i 8 yue4 11 ri4 zhu3 cha2ng 2-1 ji1 ba4i lia3o sha1n do1ng lu3 ne2ng zhi1 ho4u , lia2
True Label: Sports
Predicted Label: Sports


Title:  ka3i di2 la1 ke4 SLS sa4i we1i liu4 yue4 tui1 zu1
Content: we2i qi4ng zhu4 sha4ng ha3i to1ng yo4ng qi4 che1 che2ng li4 shi2 wu3 zho1u nia2n ,6 yue4 7 zhi4 30 ri4 , ka3i di2 la1 ke
True Label: Automobil